<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week6_Exercises_XP_Day4_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercices XP : Labo d'implémentation LoRA
Remplacez chaque `TODO` avant d'exécuter la section suivante.

## Ce que vous allez apprendre

- Les fondamentaux de LoRA (Low-Rank Adaptation) et pourquoi cela permet des réglages fins efficaces.
- Comment implémenter les matrices LoRA `A` et `B`, et comment envelopper les couches `nn.Linear` existantes.
- Les différences entre les couches linéaires standard, les couches améliorées par LoRA et les alternatives à poids fusionnés.
- Comment geler les paramètres de base pour que seuls les adaptateurs LoRA reçoivent les mises à jour.

## Ce que vous allez créer

- Un module `CoucheLoRA` réutilisable et deux wrappers linéaires (`LineaireAvecLoRA`, `LineaireAvecLoRAFusionne`).
- Un MLP à 3 couches qui peut basculer entre les variantes standard et améliorées par LoRA.
- Une boucle d'entraînement MNIST minimale et des aides à la précision pour comparer les adaptateurs gelés vs. totalement entraînables.
- Un flux de travail pour geler les poids de base et affiner uniquement les couches LoRA.

> **Point d'apprentissage**  
> Gardez les carnets de l'étudiant et de l'enseignant ouverts côte à côte. Suivez les exercices numérotés, exécutez la configuration une seule fois, et observez les formes des tenseurs au fur et à mesure que vous ajoutez les adaptateurs LoRA.

# Partie 0 : Configuration de l'environnement

Installez la pile PyTorch pour CPU ainsi que torchvision pour MNIST. Réutilisez les caches lors des réexécutions pour gagner du temps.

In [16]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [3]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

GRAINE_BASE = 123
torch.manual_seed(GRAINE_BASE)

APPAREIL = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {APPAREIL}")

Appareil utilisé : cuda


# Exercice 1 : Implémenter `CoucheLoRA`

Créez les matrices de faible rang `A` et `B`, mettez-les à l'échelle avec `alpha`, et testez le module sur un tenseur jouet.

In [5]:
class CoucheLoRA(nn.Module):
    def __init__(self, dim_entree, dim_sortie, rang, alpha):
        super().__init__()
        ecart_type = 1 / torch.sqrt(torch.tensor(rang).float())
        self.A = nn.Parameter(torch.randn(dim_entree, rang) * ecart_type)
        self.B = nn.Parameter(torch.zeros(rang, dim_sortie))
        self.alpha = alpha

    def forward(self, x):
        # Calcul de la mise à jour : (x @ A @ B) * (alpha / rang)
        correction = (x @ self.A @ self.B) * (self.alpha / self.A.shape[1])
        return correction

# Hyperparamètres pour le test
graine_aleatoire = 123
dim_entree = 784 # 28*28 pixels
dim_sortie = 10  # 10 classes (chiffres 0-9)
rang = 4
alpha = 8

torch.manual_seed(graine_aleatoire)
couche = CoucheLoRA(dim_entree, dim_sortie, rang, alpha)
x = torch.randn(1, dim_entree) # Un lot d'une image factice

print(f"Forme de l'entrée : {x.shape}")
print(couche)
print("Sortie LoRA (doit être proche de 0 car B est initialisé à 0) :", couche(x))

Forme de l'entrée : torch.Size([1, 784])
CoucheLoRA()
Sortie LoRA (doit être proche de 0 car B est initialisé à 0) : tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], grad_fn=<MulBackward0>)


# Exercice 2 : Envelopper `nn.Linear` avec LoRA

Combinez une projection linéaire gelée et une `CoucheLoRA` entraînable. Confirmez que les sorties de l'adaptateur s'ajoutent aux logits de base.

In [6]:
class LineaireAvecLoRA(nn.Module):
    def __init__(self, lineaire, rang, alpha):
        super().__init__()
        self.lineaire = lineaire
        self.lora = CoucheLoRA(
            lineaire.in_features,
            lineaire.out_features,
            rang,
            alpha,
        )

    def forward(self, x):
        # La sortie est la somme du chemin linéaire standard et du chemin LoRA
        return self.lineaire(x) + self.lora(x)

# Initialisation pour le test
lineaire_base = nn.Linear(dim_entree, dim_sortie)
# On enveloppe la couche de base avec nos paramètres de rang et alpha
couche_lora_1 = LineaireAvecLoRA(lineaire_base, rang=rang, alpha=alpha)

# Test de la sortie
sortie = couche_lora_1(x)
print(f"Forme de la sortie : {sortie.shape}")
print("Sortie LineaireAvecLoRA :", sortie)

Forme de la sortie : torch.Size([1, 10])
Sortie LineaireAvecLoRA : tensor([[-0.1714,  0.0743, -0.9784,  0.9362, -0.1236, -0.5023, -0.1977, -0.5343,
          0.4482,  0.6127]], grad_fn=<AddBackward0>)


# Exercice 3 : Échanger une couche réseau simple avec LoRA

Partez d'un perceptron à une seule couche, puis remplacez son bloc linéaire par `LineaireAvecLoRA`. Les sorties devraient correspondre avant l'entraînement car les adaptateurs LoRA commencent à zéro.

In [7]:
class ReseauUneCouche(nn.Module):
    def __init__(self, nb_features, nb_classes):
        super().__init__()
        self.couche = nn.Linear(nb_features, nb_classes)

    def forward(self, x):
        return self.couche(x)

# Initialisation avec les dimensions MNIST
reseau_simple = ReseauUneCouche(nb_features=dim_entree, nb_classes=dim_sortie)
entree_test = torch.randn(1, dim_entree)

with torch.no_grad():
    sortie_reference = reseau_simple(entree_test)

# Remplacement de la couche par LineaireAvecLoRA
reseau_simple.couche = LineaireAvecLoRA(reseau_simple.couche, rang=rang, alpha=alpha)

with torch.no_grad():
    sortie_lora = reseau_simple(entree_test)

# Vérification : les deux sorties doivent être quasi identiques (tolérance numérique)
correspondance = torch.allclose(sortie_reference, sortie_lora, atol=1e-6)
print(f"Les sorties correspondent avant l'entraînement ? {correspondance}")

Les sorties correspondent avant l'entraînement ? True


# Exercice 4 : Couche LoRA à poids fusionnés

Fusionnez les matrices LoRA avec les poids gelés pour créer une couche linéaire de remplacement qui se comporte exactement comme `LineaireAvecLoRA`.

In [8]:
class LineaireAvecLoRAFusionne(nn.Module):
    def __init__(self, lineaire, rang, alpha):
        super().__init__()
        self.lineaire = lineaire
        self.lora = CoucheLoRA(
            lineaire.in_features,
            lineaire.out_features,
            rang,
            alpha,
        )

    def forward(self, x):
        # Calcul de la matrice de poids LoRA : (A @ B).T car PyTorch utilise (entrée @ poids.T)
        lora_poids = (self.lora.A @ self.lora.B).T
        # Fusion : W_nouveau = W_original + (alpha/rang) * Delta_W
        poids_combines = self.lineaire.weight + (self.lora.alpha / self.lora.A.shape[1]) * lora_poids
        return F.linear(x, poids_combines, self.lineaire.bias)

# Test de la version fusionnée
couche_lora_2 = LineaireAvecLoRAFusionne(lineaire_base, rang=rang, alpha=alpha)
sortie_fusionnee = couche_lora_2(x)
print(f"Forme de la sortie fusionnée : {sortie_fusionnee.shape}")
print("Sortie LoRA fusionnée :", sortie_fusionnee)

Forme de la sortie fusionnée : torch.Size([1, 10])
Sortie LoRA fusionnée : tensor([[-0.1714,  0.0743, -0.9784,  0.9362, -0.1236, -0.5023, -0.1977, -0.5343,
          0.4482,  0.6127]], grad_fn=<AddmmBackward0>)


# Exercice 5 : Construire un MLP et préparer MNIST

Empilez trois couches linéaires avec des activations ReLU, puis configurez les chargeurs MNIST ainsi que l'optimiseur/état pour le pré-entraînement.

In [9]:
class PerceptronMulticouche(nn.Module):
    def __init__(self, nb_features, nb_caches_1, nb_caches_2, nb_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(nb_features, nb_caches_1),
            nn.ReLU(),
            nn.Linear(nb_caches_1, nb_caches_2),
            nn.ReLU(),
            nn.Linear(nb_caches_2, nb_classes),
        )

    def forward(self, x):
        # Aplatir l'image (batch, 1, 28, 28) -> (batch, 784)
        x = x.view(x.size(0), -1)
        x = self.layers(x)
        return x

In [10]:
# Architecture
nb_features = 784
nb_caches_1 = 128
nb_caches_2 = 64
nb_classes = 10

# Paramètres
taux_apprentissage = 0.001
nb_epoques = 3

modele = PerceptronMulticouche(
    nb_features=nb_features,
    nb_caches_1=nb_caches_1,
    nb_caches_2=nb_caches_2,
    nb_classes=nb_classes,
)

modele.to(APPAREIL)
optimiseur_preentraine = torch.optim.Adam(modele.parameters(), lr=taux_apprentissage)

print(f"Utilisation de : {APPAREIL}")
print(modele)
print("Optimiseur configuré.")

Utilisation de : cuda
PerceptronMulticouche(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
Optimiseur configuré.


## Chargement du jeu de données

In [11]:
TAILLE_LOT = 64

dataset_entrainement = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)
dataset_test = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor())

chargeur_entrainement = DataLoader(dataset=dataset_entrainement, batch_size=TAILLE_LOT, shuffle=True)
chargeur_test = DataLoader(dataset=dataset_test, batch_size=TAILLE_LOT, shuffle=False)

for images, labels in chargeur_entrainement:
    print('Dimensions du lot d\'images :', images.shape)
    print('Dimensions des étiquettes :', labels.shape)
    break

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.58MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 133kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.07MB/s]

Dimensions du lot d'images : torch.Size([64, 1, 28, 28])
Dimensions des étiquettes : torch.Size([64])


## Définir l'évaluation

In [12]:
def calculer_precision(modele, chargeur_donnees, appareil):
    modele.eval()
    preds_correctes, nb_exemples = 0, 0
    with torch.no_grad():
        for features, cibles in chargeur_donnees:
            features = features.to(appareil)
            cibles = cibles.to(appareil)
            logits = modele(features)
            _, etiquettes_predites = torch.max(logits, 1)
            nb_exemples += cibles.size(0)
            preds_correctes += (etiquettes_predites == cibles).sum()
    return (preds_correctes.float() / nb_exemples) * 100

## Entraînement

In [13]:
def entrainer(nb_epoques, modele, optimiseur, chargeur_entrainement, appareil):
    debut = time.time()
    for epoque in range(nb_epoques):
        modele.train()
        for idx_lot, (features, cibles) in enumerate(chargeur_entrainement):
            features = features.to(appareil)
            cibles = cibles.to(appareil)

            logits = modele(features)
            perte = F.cross_entropy(logits, cibles)

            optimiseur.zero_grad()
            perte.backward()
            optimiseur.step()

            if not idx_lot % 400:
                print('Époque: %03d/%03d|Lot %03d/%03d| Perte: %.4f' % (epoque+1, nb_epoques, idx_lot, len(chargeur_entrainement), perte))

        with torch.set_grad_enabled(False):
            prec = calculer_precision(modele, chargeur_entrainement, appareil)
            print('Époque: %03d/%03d précision entraînement: %.2f%%' % (epoque+1, nb_epoques, prec))

        print('Temps écoulé: %.2f min' % ((time.time() - debut)/60))
    print('Temps total d\'entraînement: %.2f min' % ((time.time() - debut)/60))

In [14]:
entrainer(nb_epoques, modele, optimiseur_preentraine, chargeur_entrainement, APPAREIL)
print(f'Précision test : {calculer_precision(modele, chargeur_test, APPAREIL):.2f}%')

/tmp/ipykernel_2022/1675484588.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Époque: %03d/%03d|Lot %03d/%03d| Perte: %.4f' % (epoque+1, nb_epoques, idx_lot, len(chargeur_entrainement), perte))


Époque: 001/003|Lot 000/938| Perte: 2.2937
Époque: 001/003|Lot 400/938| Perte: 0.4145
Époque: 001/003|Lot 800/938| Perte: 0.1156
Époque: 001/003 précision entraînement: 95.28%
Temps écoulé: 0.24 min
Époque: 002/003|Lot 000/938| Perte: 0.2208
Époque: 002/003|Lot 400/938| Perte: 0.1499
Époque: 002/003|Lot 800/938| Perte: 0.2582
Époque: 002/003 précision entraînement: 97.03%
Temps écoulé: 0.47 min
Époque: 003/003|Lot 000/938| Perte: 0.0918
Époque: 003/003|Lot 400/938| Perte: 0.0583
Époque: 003/003|Lot 800/938| Perte: 0.0335
Époque: 003/003 précision entraînement: 97.63%
Temps écoulé: 0.70 min
Temps total d'entraînement: 0.70 min
Précision test : 96.78%


# Remplacement des couches linéaires par des couches LoRA

In [15]:
modele_lora = copy.deepcopy(modele)

# Remplacement des couches linéaires 0, 2 et 4 par des versions LoRA fusionnées
modele_lora.layers[0] = LineaireAvecLoRAFusionne(modele_lora.layers[0], rang=4, alpha=8)
modele_lora.layers[2] = LineaireAvecLoRAFusionne(modele_lora.layers[2], rang=4, alpha=8)
modele_lora.layers[4] = LineaireAvecLoRAFusionne(modele_lora.layers[4], rang=4, alpha=8)

modele_lora.to(APPAREIL)

print("Modèle LoRA configuré.")
print(f'Précision test modèle original : {calculer_precision(modele, chargeur_test, APPAREIL):.2f}%')
print(f'Précision test modèle LoRA (avant tuning) : {calculer_precision(modele_lora, chargeur_test, APPAREIL):.2f}%')

Modèle LoRA configuré.
Précision test modèle original : 96.78%
Précision test modèle LoRA (avant tuning) : 96.78%


## Geler les couches linéaires originales

In [16]:
def geler_couches_lineaires(modele):
    for enfant in modele.children():
        if isinstance(enfant, nn.Linear):
            for param in enfant.parameters():
                param.requires_grad = False
        else:
            geler_couches_lineaires(enfant)

geler_couches_lineaires(modele_lora)
for nom, param in modele_lora.named_parameters():
    print(f'{nom} : {param.requires_grad}')

layers.0.lineaire.weight : False
layers.0.lineaire.bias : False
layers.0.lora.A : True
layers.0.lora.B : True
layers.2.lineaire.weight : False
layers.2.lineaire.bias : False
layers.2.lora.A : True
layers.2.lora.B : True
layers.4.lineaire.weight : False
layers.4.lineaire.bias : False
layers.4.lora.A : True
layers.4.lora.B : True


In [17]:
optimiseur_lora = torch.optim.Adam(modele_lora.parameters(), lr=taux_apprentissage)
entrainer(nb_epoques, modele_lora, optimiseur_lora, chargeur_entrainement, APPAREIL)
print(f'Précision test LoRA finetune : {calculer_precision(modele_lora, chargeur_test, APPAREIL):.2f}%')

print(f'Précision modèle original : {calculer_precision(modele, chargeur_test, APPAREIL):.2f}%')
print(f'Précision modèle LoRA : {calculer_precision(modele_lora, chargeur_test, APPAREIL):.2f}%')

Époque: 001/003|Lot 000/938| Perte: 0.0439
Époque: 001/003|Lot 400/938| Perte: 0.1236
Époque: 001/003|Lot 800/938| Perte: 0.0610
Époque: 001/003 précision entraînement: 98.23%
Temps écoulé: 0.25 min
Époque: 002/003|Lot 000/938| Perte: 0.0566
Époque: 002/003|Lot 400/938| Perte: 0.0188
Époque: 002/003|Lot 800/938| Perte: 0.0257
Époque: 002/003 précision entraînement: 98.35%
Temps écoulé: 0.50 min
Époque: 003/003|Lot 000/938| Perte: 0.0152
Époque: 003/003|Lot 400/938| Perte: 0.0440
Époque: 003/003|Lot 800/938| Perte: 0.1225
Époque: 003/003 précision entraînement: 98.34%
Temps écoulé: 0.74 min
Temps total d'entraînement: 0.74 min
Précision test LoRA finetune : 97.50%
Précision modèle original : 96.78%
Précision modèle LoRA : 97.50%


### Analyse de l'efficacité des paramètres

Comparons maintenant le nombre de paramètres que nous avons réellement entraînés avec LoRA par rapport au modèle complet.

In [18]:
def compter_parametres(modele):
    total = sum(p.numel() for p in modele.parameters())
    entrainables = sum(p.numel() for p in modele.parameters() if p.requires_grad)
    return total, entrainables

total_orig, train_orig = compter_parametres(modele)
total_lora, train_lora = compter_parametres(modele_lora)

print(f"Modèle Original - Total: {total_orig:,} | Entraînables: {train_orig:,}")
print(f"Modèle LoRA     - Total: {total_lora:,} | Entraînables: {train_lora:,}")
print(f"\nRéduction des paramètres entraînables : {100 * (1 - train_lora/train_orig):.2f}%")

Modèle Original - Total: 109,386 | Entraînables: 109,386
Modèle LoRA     - Total: 114,098 | Entraînables: 4,712

Réduction des paramètres entraînables : 95.69%


### Conclusion

Comme vous pouvez le voir, LoRA permet d'obtenir des performances similaires (voire meilleures ici grâce à une forme de régularisation) en n'entraînant qu'une infime fraction des paramètres originaux. C'est cette technique qui permet aujourd'hui de spécialiser des modèles de langage géants (LLMs) sur du matériel grand public.